# Task5 — Conditional Quantile Residuals and ECDF → UMAP → HDBSCAN

Two prespecified, unlabeled experiments. Run All reads saved artifacts by default. No Kaggle upload and no combined submission CSV. Model scores, cluster noise, and agreement are not measured anomaly F₃.

## 1. Setup

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Task5/models/residual_cluster_followup.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within CPE342-Kaggle")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from Task5.models import residual_cluster_followup as experiment
OUT = experiment.OUT
RUN_TRAINING = False
RUN_VERIFICATION = False
print(OUT)

/Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/Task5/models/results/residual_cluster_v1


## 2. Prespecified recipes

Conditional residual: 10 targets × 3 quantiles × 5 folds = 150 LightGBM regressors. Inputs contain only periods 1–3 and static context. The estimated background comes from the saved unlabeled cohort.

Clustering: 30 level + 28 shift features → median imputation → uniform QuantileTransformer (ECDF approximation), plus missing indicators → UMAP with 10 components → HDBSCAN. Three fixed UMAP seeds; median outlier percentile. A direct ECDF → HDBSCAN fit controls for UMAP. All clustering preprocessing is transductive, not OOF evaluation.

In [2]:
experiment.CONFIG

{'folds': 5,
 'fold_seed': 442,
 'quantiles': [0.1, 0.5, 0.9],
 'residual_targets': ['skill_rating',
  'skill_trajectory',
  'currency_transfers',
  'purchase_amount_mean',
  'communication_frequency',
  'friend_adds',
  'friend_removes',
  'device_count',
  'login_count',
  'reaction_time_mean'],
 'quantile_params': {'n_estimators': 250,
  'num_leaves': 15,
  'learning_rate': 0.04,
  'min_child_samples': 80,
  'reg_lambda': 3.0,
  'n_jobs': 4,
  'verbosity': -1,
  'random_state': 342},
 'quantile_transform': {'n_quantiles': 1024,
  'output_distribution': 'uniform',
  'subsample': None,
  'random_state': 342},
 'umap': {'n_components': 10,
  'n_neighbors': 30,
  'min_dist': 0.1,
  'metric': 'euclidean',
  'n_epochs': 200,
  'n_jobs': 1,
  'low_memory': True},
 'seeds': [342, 343, 344],
 'hdbscan': {'min_cluster_size': 100,
  'min_samples': 20,
  'metric': 'euclidean',
  'cluster_selection_method': 'eom',
  'core_dist_n_jobs': 4},
 'threads': 4,
 'policy': 'Each candidate has the baseli

## 3. Train or load conditional quantile residuals

In [3]:
if RUN_TRAINING:
    experiment.initialize()
    experiment.train_residual()
residual_scores = pd.read_parquet(OUT / "quantile_residual_scores.parquet")
residual_scores.head()

,test_row_index,player_id,skill_rating_conditional_score,skill_rating_q10,skill_rating_q50,skill_rating_q90,skill_rating_observed,skill_trajectory_conditional_score,skill_trajectory_q10,skill_trajectory_q50,...,login_count_q50,login_count_q90,login_count_observed,reaction_time_mean_conditional_score,reaction_time_mean_q10,reaction_time_mean_q50,reaction_time_mean_q90,reaction_time_mean_observed,conditional_residual_score,observed_target_count
0,0,P024743,1.437900,7.257351,7.306415,7.341116,True,0.205835,1.781008,2.000778,...,3.859268,3.979620,True,0.530872,5.032356,5.098573,5.178373,True,1.416037,8
1,1,P005319,0.797917,7.092411,7.153830,7.199236,True,0.433745,-1.079766,-0.945983,...,3.181101,3.329153,True,0.408298,5.628217,5.700623,5.787482,True,1.408765,10
2,2,P014970,0.029883,7.066825,7.138442,7.183592,True,0.009990,1.349767,1.542620,...,4.509489,4.645684,True,1.073618,5.071357,5.162270,5.257414,True,1.320092,10
3,3,P003275,0.129979,7.772535,7.811984,7.835476,True,0.145931,-0.907698,-0.744147,...,3.382950,3.520106,True,0.913123,5.244577,5.313087,5.389441,True,1.370511,10
4,4,P009022,0.620351,7.170102,7.235987,7.284764,True,1.372416,-1.608728,-1.383861,...,4.130952,4.281982,True,0.692796,5.234017,5.308251,5.393340,True,1.603989,9


## 4. Train or load ECDF → UMAP → HDBSCAN

HDBSCAN noise label −1 is not a confirmed anomaly. Ranking uses GLOSH outlier_scores_, not a rule that labels all noise as anomalous.

In [4]:
if RUN_TRAINING:
    experiment.initialize()
    experiment.train_clusters()
cluster_scores = pd.read_parquet(OUT / "cluster_scores.parquet")
pd.DataFrame(json.loads((OUT / "cluster_training_log.json").read_text()))

,name,seconds,clusters,noise_rows,score_distinct_values
0,ecdf_direct,18.124634,0,25889,25619
1,umap_seed342,29.740925,5,655,24190
2,umap_seed343,12.873596,2,0,24193
3,umap_seed344,13.050536,2,0,24136


## 5. Matched-budget comparison

Each of the two standalone candidates keeps 2,046 alerts, matching the current baseline, to compare ranking at a fixed budget. This is an experimental control, not an estimate of the true anomaly count. Agreement and stability cannot rank fraud accuracy.

In [5]:
if RUN_TRAINING:
    experiment.summarize()
summary = json.loads((OUT / "summary.json").read_text())
display(pd.DataFrame(summary["comparisons"]))
display(pd.DataFrame(summary["seed_sensitivity"]))
print("Measured anomaly F3:", summary["anomaly_f3"])

,method,alerts,common_with_baseline,added,removed,jaccard_vs_baseline,spearman_vs_baseline
0,conditional_quantile,2046,1172,874,874,0.401370,0.239730
1,umap_hdbscan,2046,128,1918,1918,0.032291,0.008587


,seed,matched_budget,jaccard_vs_median_seed_candidate
0,342,2046,0.644034
1,343,2046,0.734633
2,344,2046,0.724399


Measured anomaly F3: None


## 6. Verification

Reload all 150 quantile models, reproduce held-out forecasts, verify train/held-row separation, reload four clustering bundles, and verify saved candidate IDs and binary decisions.

In [6]:
if RUN_VERIFICATION:
    experiment.verify()
json.loads((OUT / "verification.json").read_text())

{'quantile_models_reloaded': 150,
 'cluster_bundles_reloaded': 4,
 'heldout_forecasts_recomputed': 776670,
 'two_candidate_ids_and_decisions_verified': True,
 'cluster_preprocessing_and_saved_embeddings_verified': True,
 'anomaly_accuracy_verified': False}

## 7. Saved predictions

These are Task5-only Parquet artifacts, not Kaggle submission CSVs. No automatic recommendation is made from unlabeled agreement. The current best submission stays unchanged.

In [7]:
for name in ["conditional_quantile", "umap_hdbscan"]:
    path = OUT / f"{name}_predictions.parquet"
    frame = pd.read_parquet(path)
    print(name, len(frame), int(frame.task5.sum()), path)
    display(frame.head())

conditional_quantile 25889 2046 /Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/Task5/models/results/residual_cluster_v1/conditional_quantile_predictions.parquet


,test_row_index,player_id,id,task5,ranking_score
0,0,P024743,ANS00001,0,1.416037
1,1,P005319,ANS00002,0,1.408765
2,2,P014970,ANS00003,0,1.320092
3,3,P003275,ANS00004,0,1.370511
4,4,P009022,ANS00005,0,1.603989


umap_hdbscan 25889 2046 /Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/Task5/models/results/residual_cluster_v1/umap_hdbscan_predictions.parquet


,test_row_index,player_id,id,task5,ranking_score
0,0,P024743,ANS00001,0,0.154158
1,1,P005319,ANS00002,0,0.876975
2,2,P014970,ANS00003,0,0.094751
3,3,P003275,ANS00004,0,0.676735
4,4,P009022,ANS00005,0,0.050330
